# Vallenar 2020 — consolidación auditable de celdas originales

Este notebook fue **derivado sin modificar el código ni los outputs** del archivo `Extraer_Vs30_USGS_Validacion_2024.ipynb` (Drive ID `1sePPNPQnz2g3rC1phL8uKl5m951Bn8Mq`).

Incluye las celdas originales 2–7, correspondientes a: reconstrucción del CSV tabular, auditoría de Vs30, ejecución del modelo V5.1 congelado, diagnóstico/robustez y dominio de aplicabilidad.

**Limitación de reproducibilidad identificada:** la propia celda de reconstrucción declara que Rrup y Sa RotD50 ya habían sido calculados y auditados previamente; por tanto, este notebook no reproduce todavía desde las señales crudas ni desde la falla finita la obtención de Rrup y Sa. Esa brecha queda explícitamente documentada.


In [ ]:
# ================================================================
# RECONSTRUIR CSV FALTANTE — VALLENAR 2020
# Evento objetivo: USGS us7000bfjr
# Mw = 6.8
#
# IMPORTANTE:
# - Rrup NO se recalcula.
# - Sa RotD50 NO se recalcula.
# - Solo se reconstruye el archivo tabular que falta en Drive.
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

# ------------------------------------------------
# 1. Carpeta oficial de la tesis
# ------------------------------------------------

BASE = "/content/drive/MyDrive/Tesis/Datos"
os.makedirs(BASE, exist_ok=True)

RUTA_SALIDA = os.path.join(
    BASE,
    "Vallenar2020_10_estaciones_Rrup_RotD50_COMPLETO.csv"
)

# ------------------------------------------------
# 2. Datos congelados de la validación
# ------------------------------------------------
#
# Coordenadas:
# metadata pública de estaciones CESMD / CSN.
#
# Rrup:
# previamente calculado respecto de la falla finita USGS.
#
# Sa:
# RotD50, T = 1.0 s, amortiguamiento 5 %,
# previamente auditado contra CESMD.
#
# NO modificar estos valores después de observar predicciones.
# ------------------------------------------------

datos = {
    "Estacion": [
        "AC04",
        "GO03",
        "AC06",
        "LCO",
        "AC05",
        "AC01",
        "AC02",
        "GO04",
        "CO06",
        "CO02"
    ],

    "Latitud": [
        -28.205,
        -27.594,
        -27.357,
        -29.011,
        -28.836,
        -26.148,
        -26.836,
        -30.173,
        -30.674,
        -31.204
    ],

    "Longitud": [
        -71.074,
        -70.235,
        -70.355,
        -70.701,
        -70.274,
        -70.599,
        -69.129,
        -70.799,
        -71.635,
        -71.000
    ],

    "Mw": [
        6.8, 6.8, 6.8, 6.8, 6.8,
        6.8, 6.8, 6.8, 6.8, 6.8
    ],

    "Rrup_km": [
        29.620,
        76.449,
        76.619,
        93.606,
        101.998,
        179.229,
        203.670,
        212.752,
        271.999,
        324.443
    ],

    "Sa_RotD50_T1_g": [
        0.070684,
        0.028799,
        0.036435,
        0.027881,
        0.019018,
        0.009810,
        0.012708,
        0.025319,
        0.002727,
        0.010347
    ]
}

df = pd.DataFrame(datos)

# ------------------------------------------------
# 3. Auditorías básicas
# ------------------------------------------------

esperadas = {
    "AC04", "GO03", "AC06", "LCO", "AC05",
    "AC01", "AC02", "GO04", "CO06", "CO02"
}

assert len(df) == 10, \
    f"ERROR: se esperaban 10 estaciones y hay {len(df)}"

assert set(df["Estacion"]) == esperadas, \
    "ERROR: no coinciden las estaciones congeladas."

assert df["Latitud"].between(-45, -15).all(), \
    "ERROR: revisar latitudes."

assert df["Longitud"].between(-80, -60).all(), \
    "ERROR: revisar longitudes."

assert (df["Rrup_km"] > 0).all(), \
    "ERROR: existe algún Rrup no positivo."

assert (df["Sa_RotD50_T1_g"] > 0).all(), \
    "ERROR: existe algún Sa no positivo."

assert np.allclose(df["Mw"], 6.8), \
    "ERROR: Mw debe mantenerse congelada en 6.8."

# ------------------------------------------------
# 4. Guardar
# ------------------------------------------------

df.to_csv(
    RUTA_SALIDA,
    index=False
)

print("=" * 70)
print("CSV VALLENAR RECONSTRUIDO CORRECTAMENTE")
print("=" * 70)

print("\nArchivo:")
print(RUTA_SALIDA)

print("\nNúmero de estaciones:", len(df))

print("\nContenido:")
display(df)

print("\nComprobación:")
print("Existe:", os.path.exists(RUTA_SALIDA))
print("Tamaño:", os.path.getsize(RUTA_SALIDA), "bytes")

In [ ]:
# ================================================================
# VALLENAR 2020 — EXTRACCIÓN Y AUDITORÍA DE Vs30
# Tesis - Modelo V5.1
#
# OBJETIVOS:
# 1. Localizar automáticamente los archivos en Google Drive.
# 2. Auditar global_vs30.grd.
# 3. Determinar qué método de extracción Vs30 se utilizó en 2024:
#       - nearest neighbor
#       - interpolación lineal
# 4. Aplicar EXACTAMENTE el mismo método a Vallenar 2020.
# 5. Generar CSV de auditoría y dataset final con Vs30.
#
# IMPORTANTE:
# - NO ejecuta todavía el modelo V5.1.
# - NO modifica Rrup ni Sa.
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import unicodedata
import numpy as np
import pandas as pd
import xarray as xr

# ================================================================
# 1. CARPETA BASE
# ================================================================

BASE = "/content/drive/MyDrive/Tesis"

print("\n====================================================")
print("1. BÚSQUEDA DE ARCHIVOS")
print("====================================================")


def buscar_archivo(nombre_exacto=None, contiene=None, extensiones=None):
    """
    Busca recursivamente dentro de /content/drive/MyDrive/Tesis
    """

    encontrados = []

    for raiz, carpetas, archivos in os.walk(BASE):
        for archivo in archivos:

            if extensiones is not None:
                if not archivo.lower().endswith(
                    tuple(e.lower() for e in extensiones)
                ):
                    continue

            if nombre_exacto is not None:
                if archivo.lower() == nombre_exacto.lower():
                    encontrados.append(os.path.join(raiz, archivo))

            elif contiene is not None:
                if contiene.lower() in archivo.lower():
                    encontrados.append(os.path.join(raiz, archivo))

    return encontrados


# ------------------------------------------------
# Raster Vs30
# ------------------------------------------------

raster_busqueda = buscar_archivo(
    nombre_exacto="global_vs30.grd"
)

if len(raster_busqueda) == 0:
    raise FileNotFoundError(
        "No se encontró global_vs30.grd dentro de /MyDrive/Tesis"
    )

RUTA_GRD = raster_busqueda[0]

print("Raster encontrado:")
print(RUTA_GRD)


# ------------------------------------------------
# CSV validación 2024
# ------------------------------------------------

busqueda_2024 = buscar_archivo(
    nombre_exacto="Vs30_estaciones_validacion_2024.csv"
)

if len(busqueda_2024) == 0:
    raise FileNotFoundError(
        "No se encontró Vs30_estaciones_validacion_2024.csv"
    )

RUTA_2024 = busqueda_2024[0]

print("\nCSV validación 2024:")
print(RUTA_2024)


# ------------------------------------------------
# CSV Vallenar
# ------------------------------------------------

nombre_vallenar = "Vallenar2020_10_estaciones_Rrup_RotD50_COMPLETO.csv"

busqueda_vallenar = buscar_archivo(
    nombre_exacto=nombre_vallenar
)

# Si no encuentra el nombre exacto, buscar cualquier CSV con Vallenar
if len(busqueda_vallenar) == 0:

    candidatos = buscar_archivo(
        contiene="Vallenar",
        extensiones=[".csv"]
    )

    print("\nNo apareció el nombre exacto.")
    print("Candidatos Vallenar encontrados:")

    for i, f in enumerate(candidatos):
        print(i, ":", f)

    if len(candidatos) == 1:
        RUTA_VALLENAR = candidatos[0]

    elif len(candidatos) == 0:
        raise FileNotFoundError(
            "\nNo se encontró ningún CSV que contenga 'Vallenar'.\n"
            "Revisa si el archivo todavía no fue copiado a Google Drive."
        )

    else:
        raise RuntimeError(
            "\nHay varios CSV de Vallenar.\n"
            "Revisa la lista impresa y define manualmente RUTA_VALLENAR."
        )

else:
    RUTA_VALLENAR = busqueda_vallenar[0]


print("\nCSV Vallenar seleccionado:")
print(RUTA_VALLENAR)


# ================================================================
# 2. FUNCIONES AUXILIARES
# ================================================================

def normalizar(texto):

    texto = unicodedata.normalize(
        "NFKD", str(texto)
    ).encode(
        "ascii", "ignore"
    ).decode()

    return re.sub(
        r"[^a-z0-9]",
        "",
        texto.lower()
    )


def detectar_columna(df, candidatos):

    mapa = {
        normalizar(c): c
        for c in df.columns
    }

    for candidato in candidatos:

        nc = normalizar(candidato)

        if nc in mapa:
            return mapa[nc]

    return None


def detectar_columnas_estacion(df):

    estacion = detectar_columna(
        df,
        [
            "Estacion",
            "Estación",
            "Station",
            "Station_Name",
            "StationCode",
            "Station_Code",
            "codigo",
            "codigo_estacion"
        ]
    )

    lat = detectar_columna(
        df,
        [
            "Latitud",
            "Latitude",
            "Lat",
            "Station_Latitude_deg",
            "Station_Latitude"
        ]
    )

    lon = detectar_columna(
        df,
        [
            "Longitud",
            "Longitude",
            "Lon",
            "Lng",
            "Station_Longitude_deg",
            "Station_Longitude"
        ]
    )

    return estacion, lat, lon


# ================================================================
# 3. ABRIR RASTER
# ================================================================

print("\n====================================================")
print("2. AUDITORÍA DEL RASTER")
print("====================================================")

ds = xr.open_dataset(
    RUTA_GRD,
    decode_times=False
)

print(ds)


# ------------------------------------------------
# Detectar coordenadas
# ------------------------------------------------

def detectar_coord(ds, candidatos):

    mapa = {
        normalizar(c): c
        for c in ds.coords
    }

    for candidato in candidatos:

        nc = normalizar(candidato)

        if nc in mapa:
            return mapa[nc]

    return None


LAT_RASTER = detectar_coord(
    ds,
    ["lat", "latitude", "y"]
)

LON_RASTER = detectar_coord(
    ds,
    ["lon", "longitude", "x"]
)


if LAT_RASTER is None or LON_RASTER is None:
    raise RuntimeError(
        f"No se identificaron lat/lon.\n"
        f"Coordenadas disponibles: {list(ds.coords)}"
    )


# ------------------------------------------------
# Variable Vs30
# ------------------------------------------------

variables_2d = [
    v for v in ds.data_vars
    if ds[v].ndim == 2
]


if "z" in ds.data_vars:
    VAR_VS30 = "z"

elif len(variables_2d) == 1:
    VAR_VS30 = variables_2d[0]

else:
    raise RuntimeError(
        f"No se identificó una variable Vs30 única.\n"
        f"Variables 2D: {variables_2d}"
    )


da = ds[VAR_VS30]


print("\nCoordenada latitud :", LAT_RASTER)
print("Coordenada longitud:", LON_RASTER)
print("Variable Vs30      :", VAR_VS30)


# ------------------------------------------------
# Resolución
# ------------------------------------------------

lat_r = np.asarray(
    ds[LAT_RASTER].values,
    dtype=float
)

lon_r = np.asarray(
    ds[LON_RASTER].values,
    dtype=float
)

dlat = np.nanmedian(
    np.abs(np.diff(lat_r))
)

dlon = np.nanmedian(
    np.abs(np.diff(lon_r))
)


print("\nResolución:")
print(f"Latitud : {dlat:.10f}°")
print(f"Longitud: {dlon:.10f}°")

print(
    f"≈ {dlat*3600:.3f} × "
    f"{dlon*3600:.3f} arc-seconds"
)


print("\nExtensión raster:")

print(
    "Latitud:",
    float(np.nanmin(lat_r)),
    "a",
    float(np.nanmax(lat_r))
)

print(
    "Longitud:",
    float(np.nanmin(lon_r)),
    "a",
    float(np.nanmax(lon_r))
)


# ================================================================
# 4. FUNCIÓN DE EXTRACCIÓN Vs30
# ================================================================

# Para interpolación se ordenan las coordenadas
da_ordenado = (
    da
    .sortby(LAT_RASTER)
    .sortby(LON_RASTER)
)


def extraer_vs30(lat, lon):

    # NEAREST
    nearest = float(
        da.sel(
            {
                LAT_RASTER: float(lat),
                LON_RASTER: float(lon)
            },
            method="nearest"
        ).values
    )

    # LINEAR
    linear = float(
        da_ordenado.interp(
            {
                LAT_RASTER: float(lat),
                LON_RASTER: float(lon)
            },
            method="linear"
        ).values
    )

    return nearest, linear


# ================================================================
# 5. RECONSTRUIR MÉTODO USADO EN VALIDACIÓN 2024
# ================================================================

print("\n====================================================")
print("3. AUDITORÍA DEL MÉTODO UTILIZADO EN 2024")
print("====================================================")

df24 = pd.read_csv(RUTA_2024)

print("\nColumnas 2024:")
print(df24.columns.tolist())


STA24, LAT24, LON24 = detectar_columnas_estacion(df24)


COL_VS24 = detectar_columna(
    df24,
    [
        "Vs30",
        "Vs30_m_s",
        "Vs30_proxy",
        "Vs30_USGS",
        "Vs30_Selected_for_Analysis_m_s"
    ]
)


print("\nColumnas detectadas:")
print("Estación :", STA24)
print("Latitud  :", LAT24)
print("Longitud :", LON24)
print("Vs30     :", COL_VS24)


if LAT24 is None or LON24 is None or COL_VS24 is None:

    print("\nERROR: no se detectaron todas las columnas 2024.")
    print("\nContenido inicial del CSV:")

    display(df24.head())

    raise RuntimeError(
        "Revisar nombres de columnas del CSV 2024."
    )


# ------------------------------------------------
# Recalcular ambas alternativas
# ------------------------------------------------

nearest_24 = []
linear_24 = []


for _, fila in df24.iterrows():

    vn, vl = extraer_vs30(
        fila[LAT24],
        fila[LON24]
    )

    nearest_24.append(vn)
    linear_24.append(vl)


df24["Vs30_nearest_recalc"] = nearest_24
df24["Vs30_linear_recalc"] = linear_24


df24["error_nearest"] = (
    df24["Vs30_nearest_recalc"]
    - df24[COL_VS24]
)

df24["error_linear"] = (
    df24["Vs30_linear_recalc"]
    - df24[COL_VS24]
)


mae_nearest = np.mean(
    np.abs(df24["error_nearest"])
)

mae_linear = np.mean(
    np.abs(df24["error_linear"])
)


max_nearest = np.max(
    np.abs(df24["error_nearest"])
)

max_linear = np.max(
    np.abs(df24["error_linear"])
)


print("\n=== COMPARACIÓN CON LOS Vs30 GUARDADOS EN 2024 ===")

print(
    f"MAE nearest = {mae_nearest:.6f} m/s"
)

print(
    f"MAE linear  = {mae_linear:.6f} m/s"
)

print(
    f"Máx. error nearest = {max_nearest:.6f} m/s"
)

print(
    f"Máx. error linear  = {max_linear:.6f} m/s"
)


# ------------------------------------------------
# Determinar método
# ------------------------------------------------

if mae_nearest < mae_linear / 5:

    METODO = "nearest"

elif mae_linear < mae_nearest / 5:

    METODO = "linear"

else:

    METODO = None


print("\n====================================================")

if METODO is not None:

    print(
        "MÉTODO IDENTIFICADO PARA VALIDACIÓN 2024:"
    )

    print(
        ">>>", METODO.upper()
    )

else:

    print(
        "ATENCIÓN: no fue posible identificar "
        "inequívocamente el método."
    )

print("====================================================")


# Mostrar primeras observaciones
columnas_mostrar_24 = []

if STA24 is not None:
    columnas_mostrar_24.append(STA24)

columnas_mostrar_24 += [
    LAT24,
    LON24,
    COL_VS24,
    "Vs30_nearest_recalc",
    "Vs30_linear_recalc",
    "error_nearest",
    "error_linear"
]


display(
    df24[columnas_mostrar_24].head(15)
)


# ================================================================
# 6. CARGAR VALLENAR
# ================================================================

print("\n====================================================")
print("4. VALLENAR 2020")
print("====================================================")

dfv = pd.read_csv(
    RUTA_VALLENAR
)


print("\nFilas:", len(dfv))

print("\nColumnas:")
print(dfv.columns.tolist())


STAV, LATV, LONV = detectar_columnas_estacion(
    dfv
)


print("\nColumnas detectadas:")
print("Estación :", STAV)
print("Latitud  :", LATV)
print("Longitud :", LONV)


if STAV is None or LATV is None or LONV is None:

    display(dfv.head())

    raise RuntimeError(
        "No fue posible detectar estación/latitud/longitud."
    )


# ================================================================
# 7. AUDITORÍA DE LAS 10 ESTACIONES
# ================================================================

estaciones_esperadas = {
    "AC04",
    "GO03",
    "AC06",
    "LCO",
    "AC05",
    "AC01",
    "AC02",
    "GO04",
    "CO06",
    "CO02"
}


estaciones_csv = set(
    dfv[STAV]
    .astype(str)
    .str.strip()
)


print("\nEstaciones encontradas:")
print(sorted(estaciones_csv))


print("\nFaltantes:")
print(
    sorted(
        estaciones_esperadas
        - estaciones_csv
    )
)


print("\nAdicionales:")
print(
    sorted(
        estaciones_csv
        - estaciones_esperadas
    )
)


if len(dfv) != 10:
    raise RuntimeError(
        f"Se esperaban 10 estaciones y hay {len(dfv)}."
    )


if estaciones_csv != estaciones_esperadas:
    raise RuntimeError(
        "Los códigos no coinciden con las 10 estaciones congeladas."
    )


# ================================================================
# 8. AUDITORÍA DE COORDENADAS
# ================================================================

lat_v = pd.to_numeric(
    dfv[LATV],
    errors="coerce"
)

lon_v = pd.to_numeric(
    dfv[LONV],
    errors="coerce"
)


if lat_v.isna().any() or lon_v.isna().any():
    raise RuntimeError(
        "Existen coordenadas faltantes/no numéricas."
    )


print("\nRango coordenadas Vallenar:")

print(
    "Latitud:",
    lat_v.min(),
    "a",
    lat_v.max()
)

print(
    "Longitud:",
    lon_v.min(),
    "a",
    lon_v.max()
)


# Rango lógico amplio Chile
if not (
    lat_v.between(-45, -15).all()
    and
    lon_v.between(-80, -60).all()
):

    raise RuntimeError(
        "Coordenadas incompatibles con Chile. "
        "Revisar posible inversión lat/lon."
    )


# ================================================================
# 9. EXTRAER Vs30 VALLENAR
# ================================================================

nearest_v = []
linear_v = []


for la, lo in zip(
    lat_v,
    lon_v
):

    vn, vl = extraer_vs30(
        la,
        lo
    )

    nearest_v.append(vn)
    linear_v.append(vl)


dfv["Vs30_nearest_m_s"] = nearest_v
dfv["Vs30_linear_m_s"] = linear_v


dfv["Delta_linear_nearest_m_s"] = (
    dfv["Vs30_linear_m_s"]
    - dfv["Vs30_nearest_m_s"]
)


# ================================================================
# 10. APLICAR EL MÉTODO HISTÓRICO
# ================================================================

if METODO == "nearest":

    dfv["Vs30_m_s"] = (
        dfv["Vs30_nearest_m_s"]
    )

elif METODO == "linear":

    dfv["Vs30_m_s"] = (
        dfv["Vs30_linear_m_s"]
    )

else:

    print(
        "\nNO se creará todavía Vs30_m_s porque "
        "el método 2024 no pudo determinarse."
    )


# ================================================================
# 11. TABLA DE AUDITORÍA
# ================================================================

columnas_resultado = [
    STAV,
    LATV,
    LONV,
    "Vs30_nearest_m_s",
    "Vs30_linear_m_s",
    "Delta_linear_nearest_m_s"
]


if "Vs30_m_s" in dfv.columns:
    columnas_resultado.append(
        "Vs30_m_s"
    )


tabla = dfv[
    columnas_resultado
].copy()


print("\n====================================================")
print("RESULTADO Vs30 — VALLENAR 2020")
print("====================================================")

display(
    tabla.round(4)
)


print(
    "\nDiferencia absoluta nearest vs linear:"
)

print(
    dfv[
        "Delta_linear_nearest_m_s"
    ].abs().describe()
)


# ================================================================
# 12. GUARDAR RESULTADOS
# ================================================================

CARPETA_SALIDA = (
    "/content/drive/MyDrive/Tesis/Datos"
)

os.makedirs(
    CARPETA_SALIDA,
    exist_ok=True
)


ruta_auditoria = os.path.join(
    CARPETA_SALIDA,
    "Vallenar2020_auditoria_Vs30.csv"
)


tabla.to_csv(
    ruta_auditoria,
    index=False
)


print("\nGuardado:")
print(ruta_auditoria)


if METODO is not None:

    ruta_final = os.path.join(
        CARPETA_SALIDA,
        "Vallenar2020_10_estaciones_Rrup_RotD50_Vs30_FINAL.csv"
    )

    dfv.to_csv(
        ruta_final,
        index=False
    )

    print("\nDataset completo guardado:")
    print(ruta_final)


# ================================================================
# 13. RESUMEN FINAL
# ================================================================

print("\n====================================================")
print("AUDITORÍA TERMINADA")
print("====================================================")

print(
    "Raster:",
    RUTA_GRD
)

print(
    "Validación 2024:",
    RUTA_2024
)

print(
    "Vallenar:",
    RUTA_VALLENAR
)

print(
    "Método histórico identificado:",
    METODO
)

print(
    "\nNO se ha ejecutado todavía el modelo V5.1."
)

In [ ]:
# ================================================================
# VALLENAR 2020 — AUDITORÍA DEFINITIVA DEL MÉTODO Vs30
#
# OBJETIVO:
# 1. Leer el mismo global_vs30.grd usado en 2024.
# 2. Recalcular Vs30 de las 44 estaciones de 2024 con:
#       a) vecino más cercano
#       b) interpolación lineal
# 3. Identificar cuál reproduce el CSV histórico 2024.
# 4. Aplicar EXACTAMENTE ese método a Vallenar 2020.
#
# NO ejecuta V5.1.
# NO modifica Mw, Rrup ni Sa.
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import xarray as xr

# ================================================================
# 1. RUTAS OFICIALES
# ================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

RUTA_GRD = os.path.join(
    BASE,
    "global_vs30.grd"
)

RUTA_2024 = os.path.join(
    BASE,
    "Vs30_estaciones_validacion_2024.csv"
)

RUTA_VALLENAR = os.path.join(
    BASE,
    "Vallenar2020_10_estaciones_Rrup_RotD50_COMPLETO.csv"
)

for ruta in [RUTA_GRD, RUTA_2024, RUTA_VALLENAR]:
    assert os.path.exists(ruta), f"No existe: {ruta}"

print("=" * 70)
print("ARCHIVOS CONFIRMADOS")
print("=" * 70)
print(RUTA_GRD)
print(RUTA_2024)
print(RUTA_VALLENAR)


# ================================================================
# 2. ABRIR RASTER USGS
# ================================================================

print("\n" + "=" * 70)
print("AUDITORÍA DEL RASTER")
print("=" * 70)

ds = xr.open_dataset(
    RUTA_GRD,
    decode_times=False
)

print(ds)


# ================================================================
# 3. IDENTIFICAR COORDENADAS DEL RASTER
# ================================================================

coords = list(ds.coords)

print("\nCoordenadas disponibles:")
print(coords)

lat_candidates = [
    c for c in coords
    if c.lower() in ["lat", "latitude", "y"]
]

lon_candidates = [
    c for c in coords
    if c.lower() in ["lon", "longitude", "x"]
]

if len(lat_candidates) != 1:
    raise RuntimeError(
        f"No se pudo identificar inequívocamente latitud: {lat_candidates}"
    )

if len(lon_candidates) != 1:
    raise RuntimeError(
        f"No se pudo identificar inequívocamente longitud: {lon_candidates}"
    )

LAT = lat_candidates[0]
LON = lon_candidates[0]


# ================================================================
# 4. IDENTIFICAR VARIABLE Vs30
# ================================================================

if "z" in ds.data_vars:
    VAR = "z"
else:
    vars_2d = [
        v for v in ds.data_vars
        if ds[v].ndim == 2
    ]

    if len(vars_2d) != 1:
        raise RuntimeError(
            f"No se identificó variable Vs30 única: {vars_2d}"
        )

    VAR = vars_2d[0]

da = ds[VAR]

print("\nLatitud raster :", LAT)
print("Longitud raster:", LON)
print("Variable Vs30  :", VAR)


# ================================================================
# 5. AUDITAR EXTENSIÓN Y RESOLUCIÓN
# ================================================================

lat_raster = np.asarray(
    ds[LAT].values,
    dtype=float
)

lon_raster = np.asarray(
    ds[LON].values,
    dtype=float
)

dlat = np.nanmedian(
    np.abs(np.diff(lat_raster))
)

dlon = np.nanmedian(
    np.abs(np.diff(lon_raster))
)

print("\nExtensión:")
print(
    f"Latitud : {np.nanmin(lat_raster):.6f} "
    f"a {np.nanmax(lat_raster):.6f}"
)

print(
    f"Longitud: {np.nanmin(lon_raster):.6f} "
    f"a {np.nanmax(lon_raster):.6f}"
)

print("\nResolución aproximada:")
print(
    f"{dlat:.10f}° × {dlon:.10f}°"
)

print(
    f"{dlat*3600:.3f} × "
    f"{dlon*3600:.3f} arc-sec"
)


# ================================================================
# 6. CORRECCIÓN AUTOMÁTICA DE LONGITUD
# ================================================================

lon_min = float(np.nanmin(lon_raster))
lon_max = float(np.nanmax(lon_raster))

def adaptar_longitud(lon):
    lon = float(lon)

    # Raster 0–360
    if lon_min >= 0 and lon < 0:
        lon = lon % 360

    # Raster -180–180
    elif lon_max <= 180 and lon > 180:
        lon = ((lon + 180) % 360) - 180

    return lon


# ================================================================
# 7. PREPARAR RASTER PARA INTERPOLACIÓN
# ================================================================

da_interp = (
    da
    .sortby(LAT)
    .sortby(LON)
)


# ================================================================
# 8. FUNCIÓN DE EXTRACCIÓN
# ================================================================

def obtener_vs30(lat, lon):

    lat = float(lat)
    lon = adaptar_longitud(lon)

    # --------------------------------
    # Método A: nearest
    # --------------------------------

    v_nearest = float(
        da.sel(
            {
                LAT: lat,
                LON: lon
            },
            method="nearest"
        ).values
    )

    # --------------------------------
    # Método B: interpolación lineal
    # --------------------------------

    v_linear = float(
        da_interp.interp(
            {
                LAT: lat,
                LON: lon
            },
            method="linear"
        ).values
    )

    return v_nearest, v_linear


# ================================================================
# 9. VALIDACIÓN HISTÓRICA 2024
# ================================================================

print("\n" + "=" * 70)
print("RECONSTRUCCIÓN DEL PROCEDIMIENTO Vs30 — VALIDACIÓN 2024")
print("=" * 70)

df24 = pd.read_csv(RUTA_2024)

print("\nColumnas CSV 2024:")
print(df24.columns.tolist())

# Estas columnas ya fueron verificadas en el archivo histórico
required_2024 = [
    "code",
    "lat",
    "lon",
    "Vs30_USGS_m_s"
]

faltan = [
    c for c in required_2024
    if c not in df24.columns
]

if faltan:
    raise RuntimeError(
        f"Faltan columnas en CSV 2024: {faltan}"
    )


# ================================================================
# 10. RECALCULAR 2024
# ================================================================

nearest = []
linear = []

for _, fila in df24.iterrows():

    vn, vl = obtener_vs30(
        fila["lat"],
        fila["lon"]
    )

    nearest.append(vn)
    linear.append(vl)


df24["Vs30_nearest"] = nearest
df24["Vs30_linear"] = linear

df24["Error_nearest"] = (
    df24["Vs30_nearest"]
    - df24["Vs30_USGS_m_s"]
)

df24["Error_linear"] = (
    df24["Vs30_linear"]
    - df24["Vs30_USGS_m_s"]
)


# ================================================================
# 11. MÉTRICAS DE REPRODUCCIÓN
# ================================================================

def metricas_error(error):

    error = np.asarray(error)

    return {
        "MAE": np.mean(np.abs(error)),
        "RMSE": np.sqrt(np.mean(error**2)),
        "MAX": np.max(np.abs(error))
    }


m_nearest = metricas_error(
    df24["Error_nearest"]
)

m_linear = metricas_error(
    df24["Error_linear"]
)


print("\n=== RESULTADO DE LA AUDITORÍA 2024 ===")

print("\nNEAREST")
print(
    f"MAE  = {m_nearest['MAE']:.9f} m/s"
)
print(
    f"RMSE = {m_nearest['RMSE']:.9f} m/s"
)
print(
    f"MAX  = {m_nearest['MAX']:.9f} m/s"
)

print("\nLINEAR")
print(
    f"MAE  = {m_linear['MAE']:.9f} m/s"
)
print(
    f"RMSE = {m_linear['RMSE']:.9f} m/s"
)
print(
    f"MAX  = {m_linear['MAX']:.9f} m/s"
)


# ================================================================
# 12. IDENTIFICAR MÉTODO HISTÓRICO
# ================================================================

# Tolerancia pequeña porque debería reproducir
# prácticamente exactamente el procedimiento histórico.

tol = 0.01  # m/s

coincide_nearest = (
    m_nearest["MAX"] <= tol
)

coincide_linear = (
    m_linear["MAX"] <= tol
)


if coincide_nearest and not coincide_linear:

    METODO = "nearest"

elif coincide_linear and not coincide_nearest:

    METODO = "linear"

elif coincide_nearest and coincide_linear:

    # Caso improbable: ambos prácticamente idénticos
    if m_nearest["RMSE"] <= m_linear["RMSE"]:
        METODO = "nearest"
    else:
        METODO = "linear"

else:

    METODO = None


print("\n" + "=" * 70)

if METODO is not None:

    print(
        "MÉTODO HISTÓRICO IDENTIFICADO:"
    )

    print(
        f">>> {METODO.upper()}"
    )

else:

    print(
        "MÉTODO NO IDENTIFICADO DE FORMA EXACTA."
    )

    print(
        "NO se seleccionará automáticamente "
        "ningún método."
    )

print("=" * 70)


# Mostrar comparación inicial
display(
    df24[
        [
            "code",
            "lat",
            "lon",
            "Vs30_USGS_m_s",
            "Vs30_nearest",
            "Vs30_linear",
            "Error_nearest",
            "Error_linear"
        ]
    ].head(15)
)


# ================================================================
# 13. DETENER SI NO HAY REPRODUCCIÓN
# ================================================================

if METODO is None:

    raise RuntimeError(
        "\nLa reconstrucción no reproduce exactamente "
        "los Vs30 de 2024.\n"
        "NO debemos avanzar a Vallenar hasta auditar "
        "esta discrepancia."
    )


# ================================================================
# 14. CARGAR VALLENAR
# ================================================================

print("\n" + "=" * 70)
print("EXTRACCIÓN Vs30 — VALLENAR 2020")
print("=" * 70)

dfv = pd.read_csv(
    RUTA_VALLENAR
)

required_v = [
    "Estacion",
    "Latitud",
    "Longitud",
    "Mw",
    "Rrup_km",
    "Sa_RotD50_T1_g"
]

faltan = [
    c for c in required_v
    if c not in dfv.columns
]

if faltan:
    raise RuntimeError(
        f"Faltan columnas Vallenar: {faltan}"
    )

assert len(dfv) == 10


# ================================================================
# 15. EXTRAER LOS DOS MÉTODOS PARA AUDITORÍA
# ================================================================

nearest_v = []
linear_v = []

for _, fila in dfv.iterrows():

    vn, vl = obtener_vs30(
        fila["Latitud"],
        fila["Longitud"]
    )

    nearest_v.append(vn)
    linear_v.append(vl)


dfv["Vs30_nearest_m_s"] = nearest_v
dfv["Vs30_linear_m_s"] = linear_v

dfv["Delta_linear_minus_nearest"] = (
    dfv["Vs30_linear_m_s"]
    - dfv["Vs30_nearest_m_s"]
)


# ================================================================
# 16. APLICAR MÉTODO CONGELADO
# ================================================================

if METODO == "nearest":

    dfv["Vs30_m_s"] = (
        dfv["Vs30_nearest_m_s"]
    )

elif METODO == "linear":

    dfv["Vs30_m_s"] = (
        dfv["Vs30_linear_m_s"]
    )


# ================================================================
# 17. CONTROLES FÍSICOS
# ================================================================

if dfv["Vs30_m_s"].isna().any():

    raise RuntimeError(
        "Existen Vs30 faltantes."
    )

if not (
    (dfv["Vs30_m_s"] > 50)
    &
    (dfv["Vs30_m_s"] < 3000)
).all():

    raise RuntimeError(
        "Aparecen Vs30 físicamente sospechosos."
    )


# ================================================================
# 18. TABLA FINAL PARA AUDITORÍA
# ================================================================

tabla_final = dfv[
    [
        "Estacion",
        "Latitud",
        "Longitud",
        "Mw",
        "Rrup_km",
        "Vs30_nearest_m_s",
        "Vs30_linear_m_s",
        "Delta_linear_minus_nearest",
        "Vs30_m_s",
        "Sa_RotD50_T1_g"
    ]
].copy()


print("\n" + "=" * 70)
print("TABLA Vs30 — VALLENAR 2020")
print("=" * 70)

display(
    tabla_final.round(
        {
            "Latitud": 6,
            "Longitud": 6,
            "Rrup_km": 3,
            "Vs30_nearest_m_s": 3,
            "Vs30_linear_m_s": 3,
            "Delta_linear_minus_nearest": 3,
            "Vs30_m_s": 3,
            "Sa_RotD50_T1_g": 6
        }
    )
)


# ================================================================
# 19. RESUMEN Vs30
# ================================================================

print("\nResumen Vs30 seleccionado:")

print(
    dfv["Vs30_m_s"].describe()
)

print(
    "\nMáxima diferencia "
    "|linear - nearest| =",
    round(
        dfv[
            "Delta_linear_minus_nearest"
        ].abs().max(),
        3
    ),
    "m/s"
)


# ================================================================
# 20. GUARDAR AUDITORÍA
# ================================================================

RUTA_AUDITORIA = os.path.join(
    BASE,
    "Vallenar2020_auditoria_Vs30.csv"
)

RUTA_FINAL = os.path.join(
    BASE,
    "Vallenar2020_10_estaciones_Rrup_RotD50_Vs30_FINAL.csv"
)

tabla_final.to_csv(
    RUTA_AUDITORIA,
    index=False
)

dfv.to_csv(
    RUTA_FINAL,
    index=False
)


print("\nArchivos guardados:")

print(
    RUTA_AUDITORIA
)

print(
    RUTA_FINAL
)


# ================================================================
# 21. MATRIZ QUE EVENTUALMENTE RECIBIRÁ V5.1
# ================================================================

X_VALLENAR = dfv[
    [
        "Mw",
        "Rrup_km",
        "Vs30_m_s"
    ]
].copy()

X_VALLENAR.columns = [
    "Mw",
    "Rrup",
    "Vs30"
]


print("\n" + "=" * 70)
print("MATRIZ DE PREDICTORES PREPARADA")
print("TODAVÍA NO ENVIADA AL MODELO V5.1")
print("=" * 70)

display(
    pd.concat(
        [
            dfv[["Estacion"]],
            X_VALLENAR
        ],
        axis=1
    ).round(3)
)

print(
    "\nV5.1 NO HA SIDO EJECUTADO."
)

In [ ]:
# =====================================================================
# VALIDACIÓN EXTERNA VALLENAR 2020 — MODELO V5.1 CONGELADO
#
# Modelo oficial:
# GradientBoostingRegressor
# Predictores: Mw + Rrup + Vs30
#
# IMPORTANTE:
# - NO reoptimiza hiperparámetros
# - NO selecciona variables
# - NO modifica estaciones
# - NO modifica Mw, Rrup, Vs30 ni Sa observado
#
# Antes de Vallenar:
# 1. reconstruye exactamente la partición oficial;
# 2. reconstruye V5.1;
# 3. reproduce el test interno oficial;
# 4. solo entonces predice Vallenar.
# =====================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import json
import hashlib
import platform
import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# =====================================================================
# 1. CONFIGURACIÓN
# =====================================================================

SEED = 42
np.random.seed(SEED)

BASE = "/content/drive/MyDrive/Tesis/Datos"

RUTA_VALLENAR = os.path.join(
    BASE,
    "Vallenar2020_10_estaciones_Rrup_RotD50_Vs30_FINAL.csv"
)

# Carpeta para que la validación externa quede separada
SALIDA = os.path.join(
    BASE,
    "Validacion_Vallenar2020"
)

os.makedirs(SALIDA, exist_ok=True)

print("=" * 76)
print("VALIDACIÓN EXTERNA VALLENAR 2020 — V5.1")
print("=" * 76)

print("Python       :", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("SEED         :", SEED)


# =====================================================================
# 2. LOCALIZAR ARCHIVOS OFICIALES V5.1
# =====================================================================

def buscar_archivo(nombre, carpeta="/content/drive/MyDrive/Tesis"):

    coincidencias = []

    for raiz, dirs, archivos in os.walk(carpeta):
        for archivo in archivos:
            if archivo.lower() == nombre.lower():
                coincidencias.append(
                    os.path.join(raiz, archivo)
                )

    if len(coincidencias) == 0:
        raise FileNotFoundError(
            f"No se encontró: {nombre}"
        )

    if len(coincidencias) > 1:
        print(
            f"\nATENCIÓN: existen varias copias de {nombre}"
        )
        for x in coincidencias:
            print(" -", x)

        # Se prioriza Resultados_V5
        preferidas = [
            x for x in coincidencias
            if "Resultados_V5" in x
        ]

        if len(preferidas) == 1:
            return preferidas[0]

        raise RuntimeError(
            "No puede determinarse inequívocamente "
            f"qué copia usar para {nombre}."
        )

    return coincidencias[0]


RUTA_DATASET = buscar_archivo(
    "01_dataset_analitico_v5_1987.csv"
)

RUTA_PARAMS = buscar_archivo(
    "05_hiperparametros_finales.json"
)


print("\nArchivos oficiales localizados:")
print("Dataset :", RUTA_DATASET)
print("Params  :", RUTA_PARAMS)
print("Vallenar:", RUTA_VALLENAR)

assert os.path.exists(RUTA_VALLENAR), (
    f"No existe el archivo Vallenar:\n{RUTA_VALLENAR}"
)


# =====================================================================
# 3. HASH DE ARCHIVOS — TRAZABILIDAD
# =====================================================================

def sha256(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for bloque in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(bloque)

    return h.hexdigest()


print("\nSHA256:")
print("Dataset :", sha256(RUTA_DATASET))
print("Params  :", sha256(RUTA_PARAMS))
print("Vallenar:", sha256(RUTA_VALLENAR))


# =====================================================================
# 4. CARGAR HIPERPARÁMETROS CONGELADOS
# =====================================================================

with open(RUTA_PARAMS, "r") as f:
    FINAL_PARAMS = json.load(f)


PARAMS_ESPERADOS = {
    "subsample": 0.7,
    "n_estimators": 400,
    "min_samples_split": 15,
    "min_samples_leaf": 2,
    "max_depth": 2,
    "learning_rate": 0.05
}


print("\n" + "=" * 76)
print("HIPERPARÁMETROS V5.1")
print("=" * 76)

print(FINAL_PARAMS)


if FINAL_PARAMS != PARAMS_ESPERADOS:

    raise RuntimeError(
        "\nDETENIDO: los hiperparámetros encontrados "
        "no coinciden exactamente con V5.1 congelado."
    )


print("\n✓ Hiperparámetros V5.1 verificados.")


# =====================================================================
# 5. CARGAR DATASET ANALÍTICO OFICIAL
# =====================================================================

df = pd.read_csv(RUTA_DATASET)

print("\n" + "=" * 76)
print("DATASET OFICIAL V5.1")
print("=" * 76)

print("Registros :", len(df))
print("Eventos   :", df["NGAsubEQID"].nunique())
print("Estaciones:", df["NGAsubSSN"].nunique())


assert len(df) == 1987, (
    f"Se esperaban 1987 registros y existen {len(df)}."
)


columnas_necesarias = [
    "NGAsubEQID",
    "NGAsubSSN",
    "Earthquake_Name",
    "Earthquake_Magnitude",
    "ClstD_km",
    "Vs30_Selected_for_Analysis_m_s",
    "T1pt000S"
]


faltantes = [
    c for c in columnas_necesarias
    if c not in df.columns
]

if faltantes:
    raise RuntimeError(
        f"Faltan columnas oficiales: {faltantes}"
    )


# =====================================================================
# 6. RECONSTRUIR EXACTAMENTE LA PARTICIÓN V5.1
# =====================================================================

# Pisco permanece como holdout independiente
pisco_mask = (
    df["Earthquake_Name"]
    .astype(str)
    .str.contains(
        "Pisco",
        case=False,
        na=False
    )
)

ids_pisco = (
    df.loc[pisco_mask, "NGAsubEQID"]
    .dropna()
    .unique()
)

assert len(ids_pisco) == 1, (
    "No se identificó un único evento Pisco."
)

PISCO_EQID = ids_pisco[0]

df_pisco = df[
    df["NGAsubEQID"] == PISCO_EQID
].copy()

df_pool = df[
    df["NGAsubEQID"] != PISCO_EQID
].copy()


# Partición oficial
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

idx_dev, idx_test = next(
    gss.split(
        df_pool,
        groups=df_pool["NGAsubEQID"]
    )
)

df_dev = (
    df_pool
    .iloc[idx_dev]
    .copy()
    .reset_index(drop=True)
)

df_test = (
    df_pool
    .iloc[idx_test]
    .copy()
    .reset_index(drop=True)
)


# Auditoría de fuga de eventos
assert set(
    df_dev["NGAsubEQID"]
).isdisjoint(
    set(df_test["NGAsubEQID"])
)

assert PISCO_EQID not in set(
    df_dev["NGAsubEQID"]
)

assert PISCO_EQID not in set(
    df_test["NGAsubEQID"]
)


print("\n" + "=" * 76)
print("PARTICIONES RECONSTRUIDAS")
print("=" * 76)

print(
    f"Desarrollo : {len(df_dev)} registros | "
    f"{df_dev['NGAsubEQID'].nunique()} eventos"
)

print(
    f"Test interno: {len(df_test)} registros | "
    f"{df_test['NGAsubEQID'].nunique()} eventos"
)

print(
    f"Pisco      : {len(df_pisco)} registros | "
    f"{df_pisco['NGAsubEQID'].nunique()} evento"
)


# Valores oficiales
assert len(df_dev) == 1583
assert len(df_test) == 382
assert len(df_pisco) == 22

assert df_dev["NGAsubEQID"].nunique() == 85
assert df_test["NGAsubEQID"].nunique() == 22


print("\n✓ Partición oficial reproducida.")


# =====================================================================
# 7. VARIABLES OFICIALES V5.1
# =====================================================================

FEATURES_FINAL = [
    "Earthquake_Magnitude",
    "ClstD_km",
    "Vs30_Selected_for_Analysis_m_s"
]

X_dev = df_dev[
    FEATURES_FINAL
].copy()

X_test = df_test[
    FEATURES_FINAL
].copy()

y_dev = np.log(
    df_dev["T1pt000S"].to_numpy()
)

y_test = np.log(
    df_test["T1pt000S"].to_numpy()
)


# =====================================================================
# 8. RECONSTRUIR V5.1 — SIN REOPTIMIZACIÓN
# =====================================================================

modelo_v51 = GradientBoostingRegressor(
    random_state=SEED,
    **FINAL_PARAMS
)

modelo_v51.fit(
    X_dev,
    y_dev
)


# =====================================================================
# 9. GUARDRAIL: REPRODUCIR TEST INTERNO OFICIAL
# =====================================================================

pred_test = modelo_v51.predict(
    X_test
)


def metricas_oficiales(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # CONVENCIÓN OFICIAL V5.1:
    # residual = predicción - observación
    resid = y_pred - y_true

    return {
        "R2": r2_score(
            y_true,
            y_pred
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        ),

        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),

        "BIAS": np.mean(resid),

        "SD_RESIDUAL": np.std(
            resid,
            ddof=1
        )
    }


m_test = metricas_oficiales(
    y_test,
    pred_test
)


OFICIAL = {
    "R2": 0.8728150622180848,
    "RMSE": 1.1516018048736865,
    "MAE": 0.8817139866066795,
    "BIAS": -0.07367153944567756,
    "SD_RESIDUAL": 1.1507500948945866
}


print("\n" + "=" * 76)
print("AUDITORÍA DEL MODELO — TEST INTERNO")
print("=" * 76)

comparacion = []

for metrica in OFICIAL:

    diferencia = (
        m_test[metrica]
        - OFICIAL[metrica]
    )

    comparacion.append({
        "Metrica": metrica,
        "Recalculado": m_test[metrica],
        "Oficial": OFICIAL[metrica],
        "Diferencia": diferencia
    })


df_comparacion = pd.DataFrame(
    comparacion
)

display(
    df_comparacion
    .round(10)
)


# Tolerancia extremadamente pequeña
for metrica in OFICIAL:

    if not np.isclose(
        m_test[metrica],
        OFICIAL[metrica],
        rtol=0,
        atol=1e-6
    ):

        raise RuntimeError(
            f"\nDETENIDO: V5.1 no reproduce "
            f"la métrica oficial {metrica}.\n"
            "NO se permite ejecutar Vallenar."
        )


print(
    "\n✓ V5.1 REPRODUCIDO CORRECTAMENTE."
)

print(
    "✓ Se autoriza la predicción externa de Vallenar."
)


# =====================================================================
# 10. CARGAR DATOS CONGELADOS DE VALLENAR
# =====================================================================

vallenar = pd.read_csv(
    RUTA_VALLENAR
)


columnas_vallenar = [
    "Estacion",
    "Mw",
    "Rrup_km",
    "Vs30_m_s",
    "Sa_RotD50_T1_g"
]


faltantes = [
    c for c in columnas_vallenar
    if c not in vallenar.columns
]

if faltantes:
    raise RuntimeError(
        f"Faltan columnas Vallenar: {faltantes}"
    )


assert len(vallenar) == 10

assert (
    vallenar["Sa_RotD50_T1_g"] > 0
).all()


# =====================================================================
# 11. MATRIZ EXTERNA EN LOS NOMBRES EXACTOS DEL MODELO
# =====================================================================

X_vallenar = pd.DataFrame({

    "Earthquake_Magnitude":
        vallenar["Mw"].to_numpy(),

    "ClstD_km":
        vallenar["Rrup_km"].to_numpy(),

    "Vs30_Selected_for_Analysis_m_s":
        vallenar["Vs30_m_s"].to_numpy()
})


assert list(
    X_vallenar.columns
) == FEATURES_FINAL


print("\n" + "=" * 76)
print("MATRIZ ENTREGADA A V5.1")
print("=" * 76)

display(
    pd.concat(
        [
            vallenar[["Estacion"]],
            X_vallenar
        ],
        axis=1
    ).round(4)
)


# =====================================================================
# 12. PREDICCIÓN VALLENAR — PRIMERA EJECUCIÓN
# =====================================================================

lnSa_obs = np.log(
    vallenar[
        "Sa_RotD50_T1_g"
    ].to_numpy()
)

lnSa_pred = modelo_v51.predict(
    X_vallenar
)

Sa_pred = np.exp(
    lnSa_pred
)


# =====================================================================
# 13. RESIDUOS
# =====================================================================

# ---------------------------------------------------------
# CONVENCIÓN OFICIAL:
# residual = predicción - observación
# BIAS > 0  -> sobreestimación promedio
# BIAS < 0  -> subestimación promedio
# ---------------------------------------------------------

residual_oficial = (
    lnSa_pred
    - lnSa_obs
)

# Solo como columna auxiliar:
residual_obs_minus_pred = (
    lnSa_obs
    - lnSa_pred
)


# =====================================================================
# 14. RESULTADO ESTACIÓN POR ESTACIÓN
# =====================================================================

resultado = vallenar[
    [
        "Estacion",
        "Latitud",
        "Longitud",
        "Mw",
        "Rrup_km",
        "Vs30_m_s",
        "Sa_RotD50_T1_g"
    ]
].copy()


resultado = resultado.rename(
    columns={
        "Sa_RotD50_T1_g":
            "Sa_obs_g"
    }
)


resultado["lnSa_obs"] = (
    lnSa_obs
)

resultado["lnSa_pred"] = (
    lnSa_pred
)

resultado["Sa_pred_g"] = (
    Sa_pred
)

resultado[
    "residual_pred_minus_obs"
] = residual_oficial

resultado[
    "residual_obs_minus_pred"
] = residual_obs_minus_pred


# Factor multiplicativo estación por estación
resultado[
    "factor_pred_obs"
] = (
    resultado["Sa_pred_g"]
    / resultado["Sa_obs_g"]
)


# =====================================================================
# 15. MÉTRICAS VALLENAR
# =====================================================================

m_vallenar = metricas_oficiales(
    lnSa_obs,
    lnSa_pred
)


# Factor asociado al BIAS
factor_bias = np.exp(
    m_vallenar["BIAS"]
)

m_vallenar[
    "FACTOR_BIAS_exp"
] = factor_bias

m_vallenar[
    "N"
] = len(vallenar)


print("\n" + "=" * 76)
print("RESULTADOS V5.1 — VALLENAR 2020")
print("=" * 76)

display(
    resultado.round(
        {
            "Latitud": 6,
            "Longitud": 6,
            "Mw": 2,
            "Rrup_km": 3,
            "Vs30_m_s": 3,
            "Sa_obs_g": 6,
            "lnSa_obs": 6,
            "lnSa_pred": 6,
            "Sa_pred_g": 6,
            "residual_pred_minus_obs": 6,
            "residual_obs_minus_pred": 6,
            "factor_pred_obs": 3
        }
    )
)


print("\n" + "=" * 76)
print("MÉTRICAS EN ln(Sa)")
print("=" * 76)

print(
    f"N     = {m_vallenar['N']}"
)

print(
    f"R²    = {m_vallenar['R2']:.6f}"
)

print(
    f"RMSE  = {m_vallenar['RMSE']:.6f}"
)

print(
    f"MAE   = {m_vallenar['MAE']:.6f}"
)

print(
    f"BIAS  = {m_vallenar['BIAS']:+.6f}"
)

print(
    f"SD residual = "
    f"{m_vallenar['SD_RESIDUAL']:.6f}"
)

print(
    f"exp(BIAS) = "
    f"{m_vallenar['FACTOR_BIAS_exp']:.4f}"
)


# =====================================================================
# 16. GUARDAR — SIN INTERPRETAR TODAVÍA
# =====================================================================

RUTA_PRED = os.path.join(
    SALIDA,
    "Vallenar2020_predicciones_V5_1.csv"
)

RUTA_MET = os.path.join(
    SALIDA,
    "Vallenar2020_metricas_V5_1.csv"
)

RUTA_AUDIT = os.path.join(
    SALIDA,
    "Vallenar2020_reproduccion_test_interno_V5_1.csv"
)


resultado.to_csv(
    RUTA_PRED,
    index=False
)

pd.DataFrame(
    [m_vallenar]
).to_csv(
    RUTA_MET,
    index=False
)

df_comparacion.to_csv(
    RUTA_AUDIT,
    index=False
)


print("\n" + "=" * 76)
print("ARCHIVOS GUARDADOS")
print("=" * 76)

print(RUTA_PRED)
print(RUTA_MET)
print(RUTA_AUDIT)


print("\n" + "=" * 76)
print("ETAPA TERMINADA")
print("=" * 76)

print(
    "El modelo V5.1 NO fue modificado."
)

print(
    "No se realizó reoptimización ni calibración."
)

print(
    "No se eliminó ninguna de las 10 estaciones."
)

print(
    "Detener aquí antes de diagnósticos y robustez."
)

In [ ]:
# =====================================================================
# VALLENAR 2020 — DIAGNÓSTICO Y ROBUSTEZ
# Modelo V5.1 ya ejecutado y congelado
#
# Este código:
# - NO entrena nuevamente el modelo
# - NO elimina estaciones
# - NO cambia Mw, Rrup, Vs30 ni Sa
#
# Analiza exclusivamente las 10 predicciones externas ya obtenidas.
# =====================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from scipy.stats import (
    pearsonr,
    spearmanr
)

# =====================================================================
# 1. RUTAS
# =====================================================================

BASE = "/content/drive/MyDrive/Tesis/Datos"

CARPETA = os.path.join(
    BASE,
    "Validacion_Vallenar2020"
)

RUTA = os.path.join(
    CARPETA,
    "Vallenar2020_predicciones_V5_1.csv"
)

assert os.path.exists(RUTA), (
    f"No existe:\n{RUTA}"
)

df = pd.read_csv(RUTA)

print("=" * 78)
print("ROBUSTEZ — VALIDACIÓN VALLENAR 2020")
print("=" * 78)

print("Registros:", len(df))
print("Estaciones:", df["Estacion"].tolist())

assert len(df) == 10


# =====================================================================
# 2. CONVENCIÓN OFICIAL
# =====================================================================

OBS = "lnSa_obs"
PRED = "lnSa_pred"
RES = "residual_pred_minus_obs"

# Comprobar consistencia del residual
res_recalc = (
    df[PRED].to_numpy()
    - df[OBS].to_numpy()
)

assert np.allclose(
    res_recalc,
    df[RES].to_numpy(),
    atol=1e-10
)

print(
    "\n✓ Convención verificada:"
    " residual = predicción - observación"
)


# =====================================================================
# 3. FUNCIÓN DE MÉTRICAS
# =====================================================================

def calcular_metricas(y_obs, y_pred):

    y_obs = np.asarray(y_obs)
    y_pred = np.asarray(y_pred)

    resid = (
        y_pred
        - y_obs
    )

    return {
        "N": len(y_obs),

        "R2": r2_score(
            y_obs,
            y_pred
        ),

        "RMSE": np.sqrt(
            mean_squared_error(
                y_obs,
                y_pred
            )
        ),

        "MAE": mean_absolute_error(
            y_obs,
            y_pred
        ),

        "BIAS": np.mean(
            resid
        ),

        "MEDIANA_RESIDUAL":
            np.median(resid),

        "SD_RESIDUAL":
            np.std(
                resid,
                ddof=1
            ),

        "FACTOR_BIAS":
            np.exp(
                np.mean(resid)
            )
    }


# =====================================================================
# 4. MÉTRICAS COMPLETAS
# =====================================================================

metricas_full = calcular_metricas(
    df[OBS],
    df[PRED]
)

print("\n" + "=" * 78)
print("MÉTRICAS — 10 ESTACIONES")
print("=" * 78)

for k, v in metricas_full.items():

    if k == "N":
        print(
            f"{k:18s}: {v}"
        )
    else:
        print(
            f"{k:18s}: {v:.6f}"
        )


# =====================================================================
# 5. RESUMEN DE SIGNO DE RESIDUOS
# =====================================================================

n_positivos = int(
    (df[RES] > 0).sum()
)

n_negativos = int(
    (df[RES] < 0).sum()
)

n_cero = int(
    np.isclose(
        df[RES],
        0
    ).sum()
)

print("\n" + "=" * 78)
print("DIRECCIÓN DE LOS ERRORES")
print("=" * 78)

print(
    f"Sobreestimaciones : "
    f"{n_positivos}/10"
)

print(
    f"Subestimaciones   : "
    f"{n_negativos}/10"
)

print(
    f"Residual mediano  : "
    f"{np.median(df[RES]):+.6f}"
)

print(
    f"Factor mediano    : "
    f"{np.exp(np.median(df[RES])):.4f}"
)


# =====================================================================
# 6. INFLUENCIA ESTACIÓN POR ESTACIÓN — JACKKNIFE
# =====================================================================

jackknife = []

for i in range(len(df)):

    temporal = (
        df
        .drop(index=i)
        .reset_index(drop=True)
    )

    m = calcular_metricas(
        temporal[OBS],
        temporal[PRED]
    )

    jackknife.append({

        "Estacion_retirada":
            df.loc[i, "Estacion"],

        "Rrup_retirado_km":
            df.loc[i, "Rrup_km"],

        "Residual_retirado":
            df.loc[i, RES],

        "R2":
            m["R2"],

        "RMSE":
            m["RMSE"],

        "MAE":
            m["MAE"],

        "BIAS":
            m["BIAS"],

        "MEDIANA_RESIDUAL":
            m["MEDIANA_RESIDUAL"],

        "SD_RESIDUAL":
            m["SD_RESIDUAL"],

        "FACTOR_BIAS":
            m["FACTOR_BIAS"]
    })


jack = pd.DataFrame(
    jackknife
)


print("\n" + "=" * 78)
print("JACKKNIFE — RETIRANDO UNA ESTACIÓN CADA VEZ")
print("=" * 78)

display(
    jack.round(6)
)


print("\nRangos jackknife:")

print(
    "R²   :",
    round(jack["R2"].min(), 4),
    "a",
    round(jack["R2"].max(), 4)
)

print(
    "RMSE :",
    round(jack["RMSE"].min(), 4),
    "a",
    round(jack["RMSE"].max(), 4)
)

print(
    "MAE  :",
    round(jack["MAE"].min(), 4),
    "a",
    round(jack["MAE"].max(), 4)
)

print(
    "BIAS :",
    round(jack["BIAS"].min(), 4),
    "a",
    round(jack["BIAS"].max(), 4)
)


# =====================================================================
# 7. ESTACIONES MÁS INFLUYENTES
# =====================================================================

jack["Cambio_RMSE"] = (
    jack["RMSE"]
    - metricas_full["RMSE"]
)

jack["Cambio_BIAS"] = (
    jack["BIAS"]
    - metricas_full["BIAS"]
)

jack["Cambio_R2"] = (
    jack["R2"]
    - metricas_full["R2"]
)


print("\n" + "=" * 78)
print("INFLUENCIA SOBRE LAS MÉTRICAS")
print("=" * 78)

display(
    jack[
        [
            "Estacion_retirada",
            "Residual_retirado",
            "Cambio_R2",
            "Cambio_RMSE",
            "Cambio_BIAS"
        ]
    ]
    .sort_values(
        "Cambio_RMSE"
    )
    .round(6)
)


# =====================================================================
# 8. BOOTSTRAP POR ESTACIONES
# =====================================================================

# Bootstrap descriptivo debido al n pequeño.
# R² se guarda, pero sus IC deben interpretarse con especial cautela.

SEED_BOOT = 20260913
B = 20000

rng = np.random.default_rng(
    SEED_BOOT
)

yobs = df[OBS].to_numpy()
ypred = df[PRED].to_numpy()

n = len(df)

bootstrap = []

for b in range(B):

    idx = rng.integers(
        0,
        n,
        size=n
    )

    o = yobs[idx]
    p = ypred[idx]

    resid = p - o

    # R² puede ser inestable con remuestreos
    # de varianza observada extremadamente baja.
    if np.var(o) > 1e-12:

        r2_b = r2_score(
            o,
            p
        )

    else:

        r2_b = np.nan

    bootstrap.append(
        [
            r2_b,

            np.sqrt(
                np.mean(
                    resid**2
                )
            ),

            np.mean(
                np.abs(resid)
            ),

            np.mean(
                resid
            ),

            np.median(
                resid
            )
        ]
    )


boot = pd.DataFrame(
    bootstrap,
    columns=[
        "R2",
        "RMSE",
        "MAE",
        "BIAS",
        "MEDIANA_RESIDUAL"
    ]
)


# =====================================================================
# 9. INTERVALOS PERCENTILES
# =====================================================================

filas_ic = []

for metrica in [
    "R2",
    "RMSE",
    "MAE",
    "BIAS",
    "MEDIANA_RESIDUAL"
]:

    serie = (
        boot[metrica]
        .dropna()
    )

    filas_ic.append({

        "Metrica":
            metrica,

        "Estimacion_original":
            (
                metricas_full["R2"]
                if metrica == "R2"

                else
                metricas_full["RMSE"]
                if metrica == "RMSE"

                else
                metricas_full["MAE"]
                if metrica == "MAE"

                else
                metricas_full["BIAS"]
                if metrica == "BIAS"

                else
                metricas_full[
                    "MEDIANA_RESIDUAL"
                ]
            ),

        "IC95_inf":
            np.percentile(
                serie,
                2.5
            ),

        "Mediana_boot":
            np.percentile(
                serie,
                50
            ),

        "IC95_sup":
            np.percentile(
                serie,
                97.5
            )
    })


ic_boot = pd.DataFrame(
    filas_ic
)


print("\n" + "=" * 78)
print("BOOTSTRAP 95 % — ESTACIONES")
print("=" * 78)

display(
    ic_boot.round(6)
)

print(
    "\nNOTA: el IC bootstrap de R² "
    "es especialmente inestable con n=10; "
    "se reporta como sensibilidad, no como "
    "evidencia inferencial fuerte."
)


# =====================================================================
# 10. TENDENCIAS DEL RESIDUAL
# =====================================================================

def correlaciones(x, y, variable):

    rp, pp = pearsonr(
        x,
        y
    )

    rs, ps = spearmanr(
        x,
        y
    )

    return {
        "Variable":
            variable,

        "Pearson_r":
            rp,

        "Pearson_p":
            pp,

        "Spearman_rho":
            rs,

        "Spearman_p":
            ps
    }


corr = pd.DataFrame(
    [
        correlaciones(
            df["Rrup_km"],
            df[RES],
            "Rrup_km"
        ),

        correlaciones(
            df["Vs30_m_s"],
            df[RES],
            "Vs30_m_s"
        )
    ]
)


print("\n" + "=" * 78)
print("RESIDUAL VS PREDICTORES")
print("=" * 78)

display(
    corr.round(6)
)

print(
    "\nLos p-values se consideran únicamente "
    "descriptivos/exploratorios por n=10 y "
    "porque las observaciones pertenecen "
    "al mismo terremoto."
)


# =====================================================================
# 11. COMPARACIÓN CON VALIDACIÓN TEMPORAL 2024
# =====================================================================

comparacion = pd.DataFrame({

    "Evento": [
        "2024 temporal externa",
        "Vallenar 2020"
    ],

    "N": [
        44,
        10
    ],

    "R2_lnSa": [
        0.8136372782,
        metricas_full["R2"]
    ],

    "RMSE_lnSa": [
        0.5731297227,
        metricas_full["RMSE"]
    ],

    "MAE_lnSa": [
        0.4647379586,
        metricas_full["MAE"]
    ],

    "BIAS_lnSa": [
        0.0420710116,
        metricas_full["BIAS"]
    ]
})


comparacion[
    "Factor_exp_BIAS"
] = np.exp(
    comparacion["BIAS_lnSa"]
)


print("\n" + "=" * 78)
print("COMPARACIÓN ENTRE EVENTOS")
print("=" * 78)

display(
    comparacion.round(6)
)


# =====================================================================
# 12. GRÁFICO 1 — OBSERVADO VS PREDICHO EN ln(Sa)
# =====================================================================

fig, ax = plt.subplots(
    figsize=(7, 7)
)

ax.scatter(
    df[OBS],
    df[PRED],
    s=55
)

lim_min = min(
    df[OBS].min(),
    df[PRED].min()
) - 0.2

lim_max = max(
    df[OBS].max(),
    df[PRED].max()
) + 0.2

ax.plot(
    [lim_min, lim_max],
    [lim_min, lim_max],
    "--"
)

for _, fila in df.iterrows():

    ax.annotate(
        fila["Estacion"],
        (
            fila[OBS],
            fila[PRED]
        ),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=9
    )

ax.set_xlim(
    lim_min,
    lim_max
)

ax.set_ylim(
    lim_min,
    lim_max
)

ax.set_xlabel(
    "ln[Sa observada (g)]"
)

ax.set_ylabel(
    "ln[Sa predicha (g)]"
)

ax.set_title(
    "Vallenar 2020 — Observado vs predicho, V5.1"
)

ax.grid(
    alpha=0.25
)

plt.tight_layout()

FIG1 = os.path.join(
    CARPETA,
    "Fig_Vallenar_lnSa_obs_vs_pred.png"
)

plt.savefig(
    FIG1,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================
# 13. GRÁFICO 2 — RESIDUAL VS Rrup
# =====================================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.scatter(
    df["Rrup_km"],
    df[RES],
    s=55
)

ax.axhline(
    0,
    linestyle="--"
)

for _, fila in df.iterrows():

    ax.annotate(
        fila["Estacion"],
        (
            fila["Rrup_km"],
            fila[RES]
        ),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=9
    )

ax.set_xlabel(
    "Rrup (km)"
)

ax.set_ylabel(
    "Residual ln(Sa): predicho − observado"
)

ax.set_title(
    "Vallenar 2020 — Residual vs Rrup"
)

ax.grid(
    alpha=0.25
)

plt.tight_layout()

FIG2 = os.path.join(
    CARPETA,
    "Fig_Vallenar_residual_vs_Rrup.png"
)

plt.savefig(
    FIG2,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================
# 14. GRÁFICO 3 — RESIDUAL VS Vs30
# =====================================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.scatter(
    df["Vs30_m_s"],
    df[RES],
    s=55
)

ax.axhline(
    0,
    linestyle="--"
)

for _, fila in df.iterrows():

    ax.annotate(
        fila["Estacion"],
        (
            fila["Vs30_m_s"],
            fila[RES]
        ),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=9
    )

ax.set_xlabel(
    "Vs30 (m/s)"
)

ax.set_ylabel(
    "Residual ln(Sa): predicho − observado"
)

ax.set_title(
    "Vallenar 2020 — Residual vs Vs30"
)

ax.grid(
    alpha=0.25
)

plt.tight_layout()

FIG3 = os.path.join(
    CARPETA,
    "Fig_Vallenar_residual_vs_Vs30.png"
)

plt.savefig(
    FIG3,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================
# 15. GRÁFICO 4 — Sa VS Rrup
# =====================================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

orden = np.argsort(
    df["Rrup_km"]
)

x = df[
    "Rrup_km"
].to_numpy()[orden]

obs_g = df[
    "Sa_obs_g"
].to_numpy()[orden]

pred_g = df[
    "Sa_pred_g"
].to_numpy()[orden]

ax.plot(
    x,
    obs_g,
    marker="o",
    label="Observado"
)

ax.plot(
    x,
    pred_g,
    marker="s",
    label="V5.1"
)

ax.set_yscale(
    "log"
)

ax.set_xlabel(
    "Rrup (km)"
)

ax.set_ylabel(
    "Sa(T=1.0 s) [g]"
)

ax.set_title(
    "Vallenar 2020 — Sa observada y predicha vs Rrup"
)

ax.legend()

ax.grid(
    alpha=0.25
)

plt.tight_layout()

FIG4 = os.path.join(
    CARPETA,
    "Fig_Vallenar_Sa_vs_Rrup.png"
)

plt.savefig(
    FIG4,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =====================================================================
# 16. GUARDAR RESULTADOS
# =====================================================================

RUTA_JACK = os.path.join(
    CARPETA,
    "Vallenar2020_jackknife_estaciones.csv"
)

RUTA_BOOT = os.path.join(
    CARPETA,
    "Vallenar2020_bootstrap_IC95.csv"
)

RUTA_CORR = os.path.join(
    CARPETA,
    "Vallenar2020_correlaciones_residual.csv"
)

RUTA_COMP = os.path.join(
    CARPETA,
    "Vallenar2020_comparacion_evento2024.csv"
)


jack.to_csv(
    RUTA_JACK,
    index=False
)

ic_boot.to_csv(
    RUTA_BOOT,
    index=False
)

corr.to_csv(
    RUTA_CORR,
    index=False
)

comparacion.to_csv(
    RUTA_COMP,
    index=False
)


print("\n" + "=" * 78)
print("ARCHIVOS GUARDADOS")
print("=" * 78)

print(RUTA_JACK)
print(RUTA_BOOT)
print(RUTA_CORR)
print(RUTA_COMP)

print(FIG1)
print(FIG2)
print(FIG3)
print(FIG4)


# =====================================================================
# 17. FIN
# =====================================================================

print("\n" + "=" * 78)
print("ROBUSTEZ TERMINADA")
print("=" * 78)

print(
    "No se modificó V5.1."
)

print(
    "No se eliminó ninguna estación."
)

print(
    "Los resultados leave-one-out son "
    "análisis de sensibilidad, no nuevas "
    "validaciones seleccionadas."
)

In [ ]:
# =====================================================================
# VALLENAR 2020 — DOMINIO DE APLICABILIDAD DE V5.1
#
# Objetivo:
# Determinar si las 10 estaciones de Vallenar se encuentran
# dentro del dominio predictor aprendido por V5.1.
#
# Se evalúa:
# 1. soporte univariado;
# 2. percentiles en desarrollo;
# 3. distancia multivariada mediante k-NN estandarizado;
# 4. comparación contra la distribución interna de desarrollo.
#
# NO modifica ni reentrena V5.1.
# =====================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# =====================================================================
# 1. RUTAS
# =====================================================================

SEED = 42

BASE = "/content/drive/MyDrive/Tesis/Datos"

RUTA_DATASET = os.path.join(
    BASE,
    "Resultados_V5",
    "01_dataset_analitico_v5_1987.csv"
)

RUTA_VALLENAR = os.path.join(
    BASE,
    "Vallenar2020_10_estaciones_Rrup_RotD50_Vs30_FINAL.csv"
)

SALIDA = os.path.join(
    BASE,
    "Validacion_Vallenar2020"
)

assert os.path.exists(RUTA_DATASET)
assert os.path.exists(RUTA_VALLENAR)

# =====================================================================
# 2. RECONSTRUIR PARTICIÓN OFICIAL
# =====================================================================

df = pd.read_csv(RUTA_DATASET)

pisco_mask = (
    df["Earthquake_Name"]
    .astype(str)
    .str.contains("Pisco", case=False, na=False)
)

ids_pisco = (
    df.loc[pisco_mask, "NGAsubEQID"]
    .dropna()
    .unique()
)

assert len(ids_pisco) == 1

PISCO_EQID = ids_pisco[0]

df_pool = df[
    df["NGAsubEQID"] != PISCO_EQID
].copy()

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

idx_dev, idx_test = next(
    gss.split(
        df_pool,
        groups=df_pool["NGAsubEQID"]
    )
)

df_dev = (
    df_pool
    .iloc[idx_dev]
    .copy()
    .reset_index(drop=True)
)

assert len(df_dev) == 1583
assert df_dev["NGAsubEQID"].nunique() == 85

print("=" * 76)
print("DOMINIO DE APLICABILIDAD — VALLENAR 2020")
print("=" * 76)

print(
    f"Desarrollo oficial: {len(df_dev)} registros, "
    f"{df_dev['NGAsubEQID'].nunique()} eventos"
)

# =====================================================================
# 3. VARIABLES
# =====================================================================

FEATURES = [
    "Earthquake_Magnitude",
    "ClstD_km",
    "Vs30_Selected_for_Analysis_m_s"
]

nombres = [
    "Mw",
    "Rrup",
    "Vs30"
]

Xdev = df_dev[
    FEATURES
].copy()

vallenar = pd.read_csv(
    RUTA_VALLENAR
)

Xval = pd.DataFrame({
    "Earthquake_Magnitude":
        vallenar["Mw"],

    "ClstD_km":
        vallenar["Rrup_km"],

    "Vs30_Selected_for_Analysis_m_s":
        vallenar["Vs30_m_s"]
})

# =====================================================================
# 4. RANGOS Y PERCENTILES DEL DESARROLLO
# =====================================================================

resumen = []

for variable, nombre in zip(
    FEATURES,
    nombres
):

    valores = Xdev[variable]

    resumen.append({

        "Variable": nombre,

        "Min_dev":
            valores.min(),

        "P01_dev":
            valores.quantile(0.01),

        "P05_dev":
            valores.quantile(0.05),

        "P50_dev":
            valores.quantile(0.50),

        "P95_dev":
            valores.quantile(0.95),

        "P99_dev":
            valores.quantile(0.99),

        "Max_dev":
            valores.max(),

        "Min_Vallenar":
            Xval[variable].min(),

        "Max_Vallenar":
            Xval[variable].max()
    })


resumen = pd.DataFrame(
    resumen
)

print("\n" + "=" * 76)
print("RANGOS DEL DOMINIO")
print("=" * 76)

display(
    resumen.round(3)
)

# =====================================================================
# 5. POSICIÓN PERCENTIL DE CADA ESTACIÓN
# =====================================================================

def percentil_empirico(
    serie,
    valor
):

    serie = np.asarray(
        serie
    )

    return (
        np.mean(
            serie <= valor
        ) * 100
    )


tabla = vallenar[
    [
        "Estacion",
        "Mw",
        "Rrup_km",
        "Vs30_m_s"
    ]
].copy()


for variable_dev, variable_v, etiqueta in [

    (
        "Earthquake_Magnitude",
        "Mw",
        "Pct_Mw"
    ),

    (
        "ClstD_km",
        "Rrup_km",
        "Pct_Rrup"
    ),

    (
        "Vs30_Selected_for_Analysis_m_s",
        "Vs30_m_s",
        "Pct_Vs30"
    )
]:

    tabla[etiqueta] = [

        percentil_empirico(
            df_dev[variable_dev],
            x
        )

        for x in vallenar[
            variable_v
        ]
    ]


# =====================================================================
# 6. BANDERAS UNIVARIADAS
# =====================================================================

tabla["Fuera_rango_Mw"] = (
    (tabla["Mw"] <
     df_dev["Earthquake_Magnitude"].min())
    |
    (tabla["Mw"] >
     df_dev["Earthquake_Magnitude"].max())
)

tabla["Fuera_rango_Rrup"] = (
    (tabla["Rrup_km"] <
     df_dev["ClstD_km"].min())
    |
    (tabla["Rrup_km"] >
     df_dev["ClstD_km"].max())
)

tabla["Fuera_rango_Vs30"] = (
    (tabla["Vs30_m_s"] <
     df_dev[
         "Vs30_Selected_for_Analysis_m_s"
     ].min())
    |
    (tabla["Vs30_m_s"] >
     df_dev[
         "Vs30_Selected_for_Analysis_m_s"
     ].max())
)

# =====================================================================
# 7. SOPORTE MULTIVARIADO — kNN
# =====================================================================
#
# Las variables tienen escalas distintas.
# Se estandarizan usando SOLO el conjunto de desarrollo.
#
# Para cada punto se calcula la distancia media a sus 5 vecinos
# más próximos en el espacio Mw-Rrup-Vs30.
#
# La distribución de referencia se obtiene dentro del propio
# desarrollo, excluyendo cada punto de sí mismo.
# =====================================================================

scaler = StandardScaler()

Xdev_z = scaler.fit_transform(
    Xdev
)

Xval_z = scaler.transform(
    Xval
)


K = 5

# Referencia interna
nn_dev = NearestNeighbors(
    n_neighbors=K + 1
)

nn_dev.fit(
    Xdev_z
)

dist_dev, _ = nn_dev.kneighbors(
    Xdev_z
)

# Primera distancia = el mismo punto (0)
dist_ref = np.mean(
    dist_dev[:, 1:],
    axis=1
)


# Vallenar
nn_val = NearestNeighbors(
    n_neighbors=K
)

nn_val.fit(
    Xdev_z
)

dist_val, _ = nn_val.kneighbors(
    Xval_z
)

distancia_knn = np.mean(
    dist_val,
    axis=1
)


tabla[
    "Distancia_media_5NN"
] = distancia_knn


# =====================================================================
# 8. UMBRALES EMPÍRICOS
# =====================================================================

P95 = np.percentile(
    dist_ref,
    95
)

P99 = np.percentile(
    dist_ref,
    99
)

print("\nReferencia interna k-NN:")

print(
    f"P95 = {P95:.4f}"
)

print(
    f"P99 = {P99:.4f}"
)


tabla[
    "Percentil_distancia_5NN"
] = [

    np.mean(
        dist_ref <= x
    ) * 100

    for x in distancia_knn
]


# Clasificación descriptiva
def clasificar(d):

    if d <= P95:
        return "Soporte habitual"

    elif d <= P99:
        return "Soporte bajo"

    else:
        return "Extrapolacion / soporte muy bajo"


tabla[
    "Clasificacion_soporte"
] = [

    clasificar(x)

    for x in tabla[
        "Distancia_media_5NN"
    ]
]


# =====================================================================
# 9. RESULTADO FINAL
# =====================================================================

print("\n" + "=" * 76)
print("SOPORTE DE LAS 10 ESTACIONES")
print("=" * 76)

display(
    tabla.round(
        {
            "Mw": 2,
            "Rrup_km": 3,
            "Vs30_m_s": 3,
            "Pct_Mw": 1,
            "Pct_Rrup": 1,
            "Pct_Vs30": 1,
            "Distancia_media_5NN": 4,
            "Percentil_distancia_5NN": 1
        }
    )
)


print("\n" + "=" * 76)
print("RESUMEN DE SOPORTE")
print("=" * 76)

print(
    tabla[
        "Clasificacion_soporte"
    ].value_counts()
)


print(
    "\nFuera de rango univariado:"
)

print(
    tabla[
        [
            "Fuera_rango_Mw",
            "Fuera_rango_Rrup",
            "Fuera_rango_Vs30"
        ]
    ].sum()
)


# =====================================================================
# 10. GUARDAR
# =====================================================================

RUTA_SOPORTE = os.path.join(
    SALIDA,
    "Vallenar2020_dominio_aplicabilidad.csv"
)

RUTA_RANGOS = os.path.join(
    SALIDA,
    "Vallenar2020_rangos_vs_desarrollo.csv"
)


tabla.to_csv(
    RUTA_SOPORTE,
    index=False
)

resumen.to_csv(
    RUTA_RANGOS,
    index=False
)


print("\nArchivos guardados:")

print(
    RUTA_SOPORTE
)

print(
    RUTA_RANGOS
)


print("\n" + "=" * 76)
print("ANÁLISIS DE DOMINIO TERMINADO")
print("=" * 76)

print(
    "No se reentrenó ni modificó V5.1."
)